# 00 · Bring your agent

The harness improves **one thing**: the text that defines your agent's behavior (a system prompt,
an `AGENTS.md`, a `SKILL.md`). It keeps that text in the MLflow Prompt Registry, hands the chosen
version to your agent on every call, and records each call as a trace.

This notebook wires up the example agent, an Acme Cloud support agent with a `lookup_policy`
tool, and shows the two pieces everything else builds on: **a versioned prompt** and **one trace
per request**. Your own agent plugs in the same way through `.env`:

| `SI_AGENT` | what you provide |
|---|---|
| `python` | `def answer(question, instructions) -> str` (any SDK) |
| `http` | a service that takes `{"input", "instructions"}` |
| `command` | a CLI, with `{input}` and `{instructions_file}` filled in |
| `litellm` | nothing: a chat model with the instructions as its system prompt |
| `openclaw` | OpenClaw, with the instructions as `AGENTS.md` (isolated profile) |

The notebooks use their own store in `.selfimprove/notebooks/`. This first one starts it fresh.

In [ ]:
import _nb as nb

experiment_id = nb.setup(fresh=True)

**Under the hood:** `nb.setup()` calls the harness's `traces.setup()`. It runs
`mlflow.set_tracking_uri()` on the SQLite file, `mlflow.create_experiment("walkthrough",
artifact_location=...)` the first time (so artifacts land in `.selfimprove/notebooks/artifacts`, not in
`./mlruns`), and `mlflow.set_experiment()`. Because the agent is the built-in example, it also calls
`mlflow.litellm.autolog()`, so live LLM calls become spans without any tracing code in the agent.

In [ ]:
import mlflow

exp = mlflow.get_experiment(experiment_id)
print("tracking uri ", mlflow.get_tracking_uri())
print("experiment   ", exp.name, "id", exp.experiment_id)
print("artifacts    ", exp.artifact_location)

## The contract

`examples/support_agent.py:answer` is all the harness needs from the example agent. When the
agent calls `lookup_policy`, that call becomes a TOOL span, and scorers use it to check whether the agent looked before it
answered.

In [ ]:
import inspect
import examples.support_agent as agent

print(inspect.getsource(agent.answer))

## 1. Register the instructions and point `production` at them (Part 2, section 1)

The agent loads its instructions **by alias**, so promoting a new version later is a pointer
move, not a code change. v1 has three seeded problems: stale quick facts (30-day refunds), a
mandatory Terms-of-Service disclaimer, and "give your best guess".

In [ ]:
import mlflow
from selfimprove import config

name = config.settings.prompt_name
v1 = mlflow.genai.register_prompt(
    name=name,
    template=open("examples/instructions.md").read(),
    commit_message="v1: initial instructions",
)
mlflow.genai.set_prompt_alias(name, alias="production", version=v1.version)

print(mlflow.genai.load_prompt(f"prompts:/{name}@production").template)

**Under the hood:** this cell is what the harness's `instructions.seed()` does (it also tags v1
`author=seed`). `mlflow.genai.register_prompt()` creates version 1 of a prompt in the Prompt Registry,
with the text and a commit message. `mlflow.genai.set_prompt_alias()` points the alias `production` at it.
An alias is a movable pointer, so later the harness ships a new version by moving it, and rolls back
by moving it again.

In [ ]:
p = mlflow.genai.load_prompt(f"prompts:/{name}@production")
print(p.uri, "| version", p.version, "| aliases", p.aliases)
print("commit message:", p.commit_message)

## 2. One traced call

`selfimprove.agents.run()` opens an AGENT span, loads the instructions **inside** the trace (which
links the prompt version to the trace), tags the trace, and calls your agent. Everything your
agent does underneath (tool spans, autologged LLM calls) nests below it.

In [ ]:
from selfimprove import agents

print(inspect.getsource(agents.run))

In [ ]:
answer = agents.run("How long can I wait to request a refund?", source="production")
print(answer)

In [ ]:
mlflow.flush_trace_async_logging()   # traces are written in the background
trace = mlflow.get_trace(mlflow.get_last_active_trace_id())
print(trace.info.tags["si_source"], "· instructions v" + trace.info.tags["si_prompt_version"])
nb.tree(trace)

**Under the hood:** `agents.run()` makes exactly one trace per request:

* `mlflow.start_span(name="my_agent", span_type=SpanType.AGENT)` opens the root span. With no trace
  active, that starts a new trace.
* `mlflow.genai.load_prompt("prompts:/walkthrough-instructions@production")` runs inside the span, so
  MLflow records the prompt version on the trace (the `mlflow.linkedPrompts` tag).
* `mlflow.update_current_trace(tags=...)` adds `si_source`, `si_prompt_version` and `si_agent`.
* Spans are exported in the background, which is why the cell above calls
  `mlflow.flush_trace_async_logging()` before `mlflow.get_trace()`.

In [ ]:
print("trace id ", trace.info.trace_id, "| state", trace.info.state)
for k in ("si_source", "si_prompt_version", "si_agent", "mlflow.linkedPrompts"):
    print(f"{k:<21}", trace.info.tags.get(k))
print("spans    ", [(s.name, s.span_type) for s in trace.data.spans])

Two tags matter for everything that follows:

* `si_prompt_version`: which version produced this answer, so signal can be sliced by version.
* `si_source`: `production` for real traffic, `eval` for evaluation, optimizer and CI calls. Only
  production traces ever count as signal, so an optimizer can never feed on its own output.

## 3. A week of traffic

`examples/questions.txt` holds the questions customers send. Replaying it twice gives the next
notebook something to look at. (`selfimprove replay examples/questions.txt` does the same from a
terminal.)

In [ ]:
questions = [q for q in open("examples/questions.txt").read().splitlines() if q and not q.startswith("#")]
for q in questions * 2:
    a = agents.run(q, source="production")
    print(f"{q[:55]:<55} -> {a[:70]!r}")

**Under the hood:** each call above wrote one more trace to the experiment. Every later step
finds production traffic with the same query: `mlflow.search_traces(filter_string="tags.si_source =
'production'")`. That is the filter the harness's `traces.production()` uses, and why the `si_source`
tag matters. `flush=True` waits for traces still being written in the background.

In [ ]:
df = mlflow.search_traces(filter_string="tags.si_source = 'production'", flush=True)
print(len(df), "production traces in the store")
df.assign(version=df["tags"].map(lambda t: t.get("si_prompt_version")),
          question=df["request"].map(lambda r: r["question"][:50]))[["trace_id", "request_time", "version", "question"]].head(5)

Open the MLflow UI on this store to browse the traces:

```bash
mlflow ui --backend-store-uri sqlite:///.selfimprove/notebooks/mlflow.db --port 5003
```

**Next:** [01 · Signal](01_signal.ipynb), where people and judges say how these answers did.